In [72]:
import csv
import json
import urllib.request

# Download the raw CSV from the root of your GitHub repository.
url = "https://raw.githubusercontent.com/jyotikeshri99999/Capstone-Project/main/appointments.csv"
csv_path = "appointments.csv"

with urllib.request.urlopen(url) as response:
    csv_text = response.read().decode("utf-8-sig")

def parse_row(row):
    """Return a copy of the row with numeric fields converted."""
    converted = row.copy()
    converted["age"] = int(converted["age"])
    converted["fee_inr"] = int(converted["fee_inr"])
    converted["prior_no_shows"] = int(converted["prior_no_shows"])
    converted["distance_km"] = float(converted["distance_km"])
    return converted

accepted = []
rejected = []

# DictReader reads the CSV text as rows of dictionaries.
reader = csv.DictReader(csv_text.splitlines())

print("Header:", reader.fieldnames)

for row in reader:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

rows_read = len(accepted) + len(rejected)

print("Rows read:", rows_read)
print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

# Save rows that could not be converted.
with open("rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

print("Saved rejected_rows.json")

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Rows read: 1020
Accepted: 988
Rejected: 32
Saved rejected_rows.json


In [73]:
import pandas as pd

url = "https://raw.githubusercontent.com/jyotikeshri99999/Capstone-Project/main/appointments.csv"

# Read the original file again; don't use Part 1.1's accepted/rejected lists.
df = pd.read_csv(url)

print("Raw shape:", df.shape)
print("\nColumn information:")
df.info()

print("\nMissing values by column:")
print(df.isnull().sum())

print("\nExact duplicate rows:", df.duplicated().sum())
print(
    "Distinct raw booking_channel values:",
    df["booking_channel"].nunique(dropna=True)
)

# (a) Drop exact duplicate rows.
df = df.drop_duplicates().copy()
print("\nRows after dropping exact duplicates:", len(df))

# (b) Clean booking_channel labels.
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print(
    "Distinct booking_channel values after cleaning:",
    df["booking_channel"].nunique(dropna=True)
)
print("Rows after cleaning booking_channel:", len(df))

# (c) Convert dates.
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("Rows after converting dates:", len(df))

# (d) Drop rows with a missing appointment date or status.
df = df.dropna(subset=["appt_date", "status"]).copy()
print("Rows after dropping missing appt_date or status:", len(df))

Raw shape: (1020, 14)

Column information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values by column:
appointment_id      0


In [74]:
import numpy as np

# (a) Mark impossible ages as missing.
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages:", int(invalid_age_mask.sum()))
df.loc[invalid_age_mask, "age"] = np.nan

# (b) Fill missing ages and distances with their medians.
print("Missing ages before filling:", int(df["age"].isna().sum()))
print("Median age:", df["age"].median())
df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)

print("Missing distances before filling:", int(df["distance_km"].isna().sum()))
print("Median distance_km:", df["distance_km"].median())
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())

# (c) Calculate IQR fences, then cap values outside the fences.
def cap_outliers(column_name):
    q1 = df[column_name].quantile(0.25)
    q3 = df[column_name].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    capped_count = int(
        ((df[column_name] < lower_fence) |
         (df[column_name] > upper_fence)).sum()
    )

    df[column_name] = np.clip(
        df[column_name],
        lower_fence,
        upper_fence
    )

    print(
        f"{column_name}: Q1={q1}, Q3={q3}, "
        f"IQR={iqr}, lower fence={lower_fence}, "
        f"upper fence={upper_fence}, values capped={capped_count}"
    )

cap_outliers("distance_km")
cap_outliers("fee_inr")

# (d) Round the capped columns as specified.
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("\nMedian-filled age:", df["age"].median())
print("Largest fee after capping:", df["fee_inr"].max())
print("Missing ages remaining:", int(df["age"].isna().sum()))
print("Missing distances remaining:", int(df["distance_km"].isna().sum()))

Invalid ages: 5
Missing ages before filling: 17
Median age: 48.0
Missing distances before filling: 20
Median distance_km: 8.25
distance_km: Q1=4.625, Q3=13.0, IQR=8.375, lower fence=-7.9375, upper fence=25.5625, values capped=26
fee_inr: Q1=500.0, Q3=787.5, IQR=287.5, lower fence=68.75, upper fence=1218.75, values capped=3

Median-filled age: 48.0
Largest fee after capping: 1219
Missing ages remaining: 0
Missing distances remaining: 0


In [75]:
# Create the two derived columns.
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

# Convert the four requested columns into a NumPy array.
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) Mean of each column.
column_means = np.mean(X, axis=0)
print("Means [age, distance_km, fee_inr, lead_days]:", column_means)

# (b) Count and share of appointments with lead_days > 14.
long_lead_mask = df["lead_days"].to_numpy() > 14
print("Appointments with lead_days > 14:", int(long_lead_mask.sum()))
print("Share with lead_days > 14:", round(long_lead_mask.mean() * 100, 1), "%")

# (c) Distance z-scores and count where absolute z-score > 2.
distance = X[:, 1]
distance_z = (distance - np.mean(distance)) / np.std(distance)
print("Rows with |distance z-score| > 2:", int((np.abs(distance_z) > 2).sum()))

# Show the overall no-show count and rate for comparison.
print("No-shows:", int(df["no_show"].sum()), "out of", len(df))
print("No-show rate:", round(df["no_show"].mean() * 100, 1), "%")

Means [age, distance_km, fee_inr, lead_days]: [ 48.17373737   9.50090909 635.71414141  11.81919192]
Appointments with lead_days > 14: 296
Share with lead_days > 14: 29.9 %
Rows with |distance z-score| > 2: 53
No-shows: 218 out of 990
No-show rate: 22.0 %


In [76]:
import requests
import pandas as pd

clinics_url = (
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/clinics.csv"
)

clinics = pd.read_csv(clinics_url)
temperatures = []

for _, clinic in clinics.iterrows():
    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temp_c = None
    last_reason = "Unknown error"

    # Try once, then retry once if the request or response fails.
    for attempt in range(2):
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params=params,
                timeout=10
            )

            if response.status_code != 200:
                last_reason = f"HTTP status {response.status_code}"
            else:
                data = response.json()
                hourly = data.get("hourly", {})
                readings = hourly.get("temperature_2m")

                if readings:
                    temp_c = readings[0]
                    break
                else:
                    last_reason = "Hourly temperature data is missing or empty"

        except requests.exceptions.RequestException as error:
            last_reason = str(error)
        except (ValueError, TypeError) as error:
            last_reason = f"Invalid response: {error}"

    if temp_c is None:
        print(f"{clinic['clinic_name']}: weather lookup failed — {last_reason}")

    temperatures.append(temp_c)

clinics["temp_c"] = temperatures
clinics.to_csv("clinics_enriched.csv", index=False)

print("\nSaved clinics_enriched.csv")
print("Rows:", len(clinics))
print(clinics[["clinic_name", "temp_c"]])


Saved clinics_enriched.csv
Rows: 8
                clinic_name  temp_c
0     Kothrud Family Clinic    19.2
1      Hadapsar Care Centre    19.5
2    Sitabuldi Health Point    18.6
3      Malviya Nagar Clinic    14.4
4     Vaishali Nagar Clinic    14.4
5  Edappally Medical Centre    27.9
6        Vijay Nagar Clinic    16.6
7    Gomti Nagar Care Point    14.0


In [77]:
import pandas as pd

columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
]

# Sort and keep the required column order.
clean = df.sort_values("appointment_id").copy()

# Format dates as YYYY-MM-DD text.
clean["booked_on"] = clean["booked_on"].dt.strftime("%Y-%m-%d")
clean["appt_date"] = clean["appt_date"].dt.strftime("%Y-%m-%d")
clean = clean[columns]

# Check the required export conditions.
print("Unique appointment IDs:", clean["appointment_id"].nunique())
print("Appointment rows:", len(clean))
print("Missing values by column:")
print(clean.isna().sum())
print("Total missing values:", int(clean.isna().sum().sum()))

# Save the clean CSV.
clean.to_csv("appointments_clean.csv", index=False)
print("\nSaved appointments_clean.csv")

def quote_sql(value):
    """Format a text value for this fixed dataset's SQL inserts."""
    return "'" + str(value).replace("'", "''") + "'"

def sql_number(value):
    """Format a numeric value, or use SQL NULL if it is missing."""
    if pd.isna(value):
        return "NULL"
    return str(value)

def sql_decimal(value, places):
    """Format a decimal value, or use SQL NULL if it is missing."""
    if pd.isna(value):
        return "NULL"
    return f"{float(value):.{places}f}"

with open("load_data.sql", "w", encoding="utf-8") as sql_file:

    # Clinic inserts must come first because appointments reference clinics.
    for _, row in clinics.iterrows():
        values = [
            quote_sql(row["clinic_id"]),
            quote_sql(row["clinic_name"]),
            quote_sql(row["city"]),
            quote_sql(row["tier"]),
            sql_decimal(row["latitude"], 4),
            sql_decimal(row["longitude"], 4),
            sql_decimal(row["temp_c"], 1)
        ]
        sql_file.write(
            "INSERT INTO clinics VALUES (" + ",".join(values) + ");\n"
        )

    # Appointment inserts.
    for _, row in clean.iterrows():
        values = [
            quote_sql(row["appointment_id"]),
            quote_sql(row["patient_id"]),
            quote_sql(row["clinic_id"]),
            quote_sql(row["booked_on"]),
            quote_sql(row["appt_date"]),
            sql_number(row["age"]),
            quote_sql(row["gender"]),
            sql_decimal(row["distance_km"], 1),
            sql_number(row["fee_inr"]),
            quote_sql(row["booking_channel"]),
            quote_sql(row["visit_type"]),
            quote_sql(row["reminder_sent"]),
            sql_number(row["prior_no_shows"]),
            quote_sql(row["status"]),
            sql_number(row["lead_days"]),
            sql_number(row["no_show"])
        ]
        sql_file.write(
            "INSERT INTO appointments VALUES (" + ",".join(values) + ");\n"
        )

print("Saved load_data.sql")
print("Expected INSERT statements:", len(clinics) + len(clean))

Unique appointment IDs: 990
Appointment rows: 990
Missing values by column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64
Total missing values: 0

Saved appointments_clean.csv
Saved load_data.sql
Expected INSERT statements: 998


In [78]:
import pandas as pd

appointments = pd.read_csv(
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/appointments_clean.csv"
)
clinics = pd.read_csv(
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/clinics_enriched.csv"
)

clinic_check = (
    appointments.groupby("clinic_id")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .merge(clinics[["clinic_id", "clinic_name", "city"]], on="clinic_id")
)

clinic_check["no_show_pct"] = (
    100 * clinic_check["no_shows"] / clinic_check["appointments"]
).round(1)

print(clinic_check.sort_values("no_show_pct", ascending=False).to_string(index=False))

clinic_id  appointments  no_shows              clinic_name    city  no_show_pct
      C08            90        30   Gomti Nagar Care Point Lucknow         33.3
      C02           143        39     Hadapsar Care Centre    Pune         27.3
      C04           142        38     Malviya Nagar Clinic  Jaipur         26.8
      C01           160        36    Kothrud Family Clinic    Pune         22.5
      C07           101        21       Vijay Nagar Clinic  Indore         20.8
      C06           142        23 Edappally Medical Centre   Kochi         16.2
      C03            82        13   Sitabuldi Health Point  Nagpur         15.9
      C05           130        18    Vaishali Nagar Clinic  Jaipur         13.8


In [79]:
import pandas as pd
from sklearn.model_selection import train_test_split

url = (
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/appointments_clean.csv"
)
appointments = pd.read_csv(url)

# Keep IDs and target aside; neither is used as a feature.
appointment_ids = appointments["appointment_id"]
y = appointments["no_show"]

# Start with the six specified numeric/binary features.
X = appointments[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent"
    ]
].copy()

# Convert reminder values to 1 and 0.
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})

# Add one-hot columns for booking channel and visit type.
X["booking_channel"] = appointments["booking_channel"]
X["visit_type"] = appointments["visit_type"]
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

# Keep the exact split required by the assignment.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Feature columns:", list(X.columns))
print("Number of features:", X.shape[1])
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("No-shows in test set:", int(y_test.sum()))

# Baseline: always predict Attended (0).
baseline_accuracy = (y_test == 0).mean()
print("Always-Attended baseline accuracy:", baseline_accuracy)
print("Baseline correct predictions:", int((y_test == 0).sum()), "of", len(y_test))

Feature columns: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Number of features: 11
Training rows: 792
Test rows: 198
No-shows in test set: 49
Always-Attended baseline accuracy: 0.7525252525252525
Baseline correct predictions: 149 of 198


In [80]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Scale the features using training data only.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Fit logistic regression and predict on the test set.
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

y_pred = lr.predict(X_test_s)
y_prob = lr.predict_proba(X_test_s)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

# Show coefficients in the feature-column order.
coefficients = pd.Series(lr.coef_[0], index=X.columns)
print("\nCoefficients:")
print(coefficients.sort_values())

# Reminder coefficient interpretation.
reminder_coef = coefficients["reminder_sent"]
reminder_index = list(X.columns).index("reminder_sent")
reminder_scale = scaler.scale_[reminder_index]

print("\nReminder coefficient:", reminder_coef)
print("Odds multiplier per 1 SD:", np.exp(reminder_coef))
print("Reminder feature SD:", reminder_scale)
print("Odds multiplier for No → Yes:", np.exp(reminder_coef / reminder_scale))

Accuracy: 0.7878787878787878
Confusion matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.6296296296296297
Recall: 0.3469387755102041
F1: 0.4473684210526316
ROC-AUC: 0.788933022873579

Coefficients:
reminder_sent             -0.580669
age                       -0.253880
booking_channel_Walk-In   -0.186364
booking_channel_App       -0.046804
visit_type_Follow-up      -0.015658
visit_type_Consultation    0.015658
booking_channel_Phone      0.215881
distance_km                0.253448
fee_inr                    0.452761
lead_days                  0.614471
prior_no_shows             0.649612
dtype: float64

Reminder coefficient: -0.5806686033659922
Odds multiplier per 1 SD: 0.559524141750829
Reminder feature SD: 0.4949752621616814
Odds multiplier for No → Yes: 0.3093980903124433


In [81]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Full-depth tree.
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

full_train_accuracy = accuracy_score(y_train, tree_full.predict(X_train))
full_test_accuracy = accuracy_score(y_test, tree_full.predict(X_test))

print("Full-depth tree")
print("Depth:", tree_full.get_depth())
print("Training accuracy:", full_train_accuracy)
print("Test accuracy:", full_test_accuracy)

# Tree limited to depth 3.
tree_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_depth3.fit(X_train, y_train)

depth3_train_accuracy = accuracy_score(y_train, tree_depth3.predict(X_train))
depth3_test_accuracy = accuracy_score(y_test, tree_depth3.predict(X_test))

print("\nDepth-3 tree")
print("Depth:", tree_depth3.get_depth())
print("Training accuracy:", depth3_train_accuracy)
print("Test accuracy:", depth3_test_accuracy)

Full-depth tree
Depth: 15
Training accuracy: 1.0
Test accuracy: 0.6717171717171717

Depth-3 tree
Depth: 3
Training accuracy: 0.821969696969697
Test accuracy: 0.7777777777777778


In [82]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
import pandas as pd

# Fit the random forest on the unscaled training features.
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_train_accuracy = accuracy_score(y_train, rf.predict(X_train))
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print("Random forest training accuracy:", rf_train_accuracy)
print("Random forest test accuracy:", rf_accuracy)
print("Random forest test recall:", rf_recall)
print("Random forest test ROC-AUC:", rf_auc)

# List the three features with the largest importance scores.
importance = pd.Series(rf.feature_importances_, index=X.columns)
print("\nTop three feature importances:")
print(importance.sort_values(ascending=False).head(3))

# Compare the models on the test set.
comparison = pd.DataFrame([
    {
        "model": "Logistic regression",
        "accuracy": accuracy_score(y_test, lr.predict(X_test_s)),
        "recall": recall_score(y_test, lr.predict(X_test_s)),
        "roc_auc": roc_auc_score(y_test, lr.predict_proba(X_test_s)[:, 1])
    },
    {
        "model": "Depth-3 decision tree",
        "accuracy": accuracy_score(y_test, tree_depth3.predict(X_test)),
        "recall": recall_score(y_test, tree_depth3.predict(X_test)),
        "roc_auc": roc_auc_score(
            y_test, tree_depth3.predict_proba(X_test)[:, 1]
        )
    },
    {
        "model": "Random forest",
        "accuracy": rf_accuracy,
        "recall": rf_recall,
        "roc_auc": rf_auc
    }
])

print("\nTest-set comparison:")
print(comparison.round(3).to_string(index=False))

Random forest training accuracy: 1.0
Random forest test accuracy: 0.7474747474747475
Random forest test recall: 0.30612244897959184
Random forest test ROC-AUC: 0.7057252431173813

Top three feature importances:
lead_days      0.219905
age            0.196788
distance_km    0.194739
dtype: float64

Test-set comparison:
                model  accuracy  recall  roc_auc
  Logistic regression     0.788   0.347    0.789
Depth-3 decision tree     0.778   0.224    0.700
        Random forest     0.747   0.306    0.706


In [83]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Logistic regression pipeline: scaling is re-fitted inside each CV fold.
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

tree3_cv_model = DecisionTreeClassifier(max_depth=3, random_state=42)
rf_cv_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Tune the forest using training data only.
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    cv=5,
    scoring="roc_auc"
)
grid.fit(X_train, y_train)

print("Tuned forest best parameters:", grid.best_params_)
print("Tuned forest best CV ROC-AUC:", grid.best_score_)

# Compare mean CV ROC-AUC scores on training data only.
lr_cv_auc = cross_val_score(
    lr_pipe, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

tree3_cv_auc = cross_val_score(
    tree3_cv_model, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

rf_cv_auc = cross_val_score(
    rf_cv_model, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

cv_scores = {
    "Logistic regression": lr_cv_auc,
    "Depth-3 tree": tree3_cv_auc,
    "Random forest": rf_cv_auc,
    "Tuned random forest": grid.best_score_
}

print("\nTraining cross-validation ROC-AUC:")
for model_name, score in cv_scores.items():
    print(f"{model_name}: {score:.3f}")

# Select the model with the highest training CV ROC-AUC.
if max(cv_scores, key=cv_scores.get) == "Logistic regression":
    winner_name = "Logistic regression"
    winner = lr_pipe
elif max(cv_scores, key=cv_scores.get) == "Depth-3 tree":
    winner_name = "Depth-3 tree"
    winner = tree3_cv_model
elif max(cv_scores, key=cv_scores.get) == "Random forest":
    winner_name = "Random forest"
    winner = rf_cv_model
else:
    winner_name = "Tuned random forest"
    winner = grid.best_estimator_

print("\nSelected model:", winner_name)
print("Selected CV ROC-AUC:", max(cv_scores.values()))

# Out-of-fold training probabilities; use these to choose a threshold.
oof_prob = cross_val_predict(
    winner,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

threshold_results = []

for threshold in [0.5, 0.35]:
    oof_flag = (oof_prob >= threshold).astype(int)
    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_train, oof_flag, zero_division=0),
        "recall": recall_score(y_train, oof_flag, zero_division=0),
        "f1": f1_score(y_train, oof_flag, zero_division=0),
        "flagged": int(oof_flag.sum())
    })

print("\nOut-of-fold training results:")
print(pd.DataFrame(threshold_results).round(3).to_string(index=False))

# Choose the threshold with the higher out-of-fold F1.
chosen_threshold = max(threshold_results, key=lambda row: row["f1"])["threshold"]
print("\nChosen threshold:", chosen_threshold)

# Fit the selected model on all training rows, then evaluate on the test set once.
winner.fit(X_train, y_train)
test_prob = winner.predict_proba(X_test)[:, 1]
test_flag = (test_prob >= chosen_threshold).astype(int)

print("\nOne test-set evaluation:")
print("Precision:", precision_score(y_test, test_flag, zero_division=0))
print("Recall:", recall_score(y_test, test_flag, zero_division=0))
print("Flagged:", int(test_flag.sum()))
print("ROC-AUC:", roc_auc_score(y_test, test_prob))

# Save a risk score for every test row.
test_risk_scores = pd.DataFrame({
    "appointment_id": appointments.loc[X_test.index, "appointment_id"].to_numpy(),
    "no_show": y_test.to_numpy(),
    "risk_prob": test_prob,
    "flag": test_flag
})
test_risk_scores.to_csv("test_risk_scores.csv", index=False)

print("\nSaved test_risk_scores.csv with", len(test_risk_scores), "rows")

Tuned forest best parameters: {'max_depth': 3, 'n_estimators': 200}
Tuned forest best CV ROC-AUC: 0.7673242251739405

Training cross-validation ROC-AUC:
Logistic regression: 0.774
Depth-3 tree: 0.678
Random forest: 0.747
Tuned random forest: 0.767

Selected model: Logistic regression
Selected CV ROC-AUC: 0.7737139899948249

Out-of-fold training results:
 threshold  precision  recall    f1  flagged
      0.50      0.575   0.249 0.347       73
      0.35      0.513   0.467 0.489      154

Chosen threshold: 0.35

One test-set evaluation:
Precision: 0.5714285714285714
Recall: 0.5714285714285714
Flagged: 49
ROC-AUC: 0.788933022873579

Saved test_risk_scores.csv with 198 rows


In [5]:
!pip install -q google-genai

import json
import pandas as pd
from getpass import getpass
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3.6-flash"

# Enter the key when prompted; it won't be echoed in the cell output.
API_KEY = getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)

SYSTEM_TEXT = """
You classify short patient follow-up notes for an outpatient clinic.
Use only the note as evidence. If the reason is unclear, choose "other".
Return the requested structured fields.
"""

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

def build_craft_prompt(note_text):
    return f"""
Context: CareLoop Clinics reviews notes after patients miss appointments.
Role: You are a careful analyst classifying the likely reason for one missed visit.
Action: Classify the note, summarize it in no more than 25 words, and suggest a practical follow-up action. If the reason is unclear, choose "other" rather than guessing.
Tone: Neutral, concise, and respectful.
Format: Return the four fields required by the JSON schema. The root_cause must be exactly one of:
forgot_or_no_reminder, distance_or_transport, cost_or_fee,
illness_or_emergency, work_or_schedule, other.

Patient note:
{note_text}
"""

def analyse_note(note_text):
    prompt = build_craft_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    return json.loads(response.text)

# If your CSV is in the repo's data/ folder, add "data/" to this URL.
notes_url = (
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/followup_notes.csv"
)
notes = pd.read_csv(notes_url)

note_n001 = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
result = analyse_note(note_n001)
print(result)

Paste your Gemini API key: ··········
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient missed her visit after receiving no reminder message and forgetting. She requested to rebook for next week.', 'recommended_action': 'Verify patient contact preferences for reminders and reschedule the appointment for next week.'}


In [6]:
!pip install -q google-genai pydantic

import json
import time
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from google.genai import types

ALLOWED_CAUSES = {
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
}

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def get_note_reply(note_text):
    """Return both the raw JSON text and its parsed dictionary."""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_craft_prompt(note_text),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )
    raw_text = response.text
    parsed_data = json.loads(raw_text)
    return raw_text, parsed_data

# Use the repo's root URL. If the CSV is in data/, add "data/" before its name.
notes_url = (
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/followup_notes.csv"
)
notes = pd.read_csv(notes_url)

results = []
api_calls = 0

for _, row in notes.head(10).iterrows():
    note_text = row["note_text"]

    # Required blank-note check: test NaN before converting to string.
    if pd.isna(note_text) or str(note_text).strip() == "":
        results.append({
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })
        continue

    # Wait between actual API calls, not before the first call.
    if api_calls > 0:
        time.sleep(13)

    api_calls += 1
    raw_response = ""
    data = {}

    try:
        raw_response, data = get_note_reply(str(note_text).strip())

        # Keep any fields the model returned, even if validation fails.
        result = {
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": data.get("root_cause", ""),
            "confidence": data.get("confidence", ""),
            "summary": data.get("summary", ""),
            "recommended_action": data.get("recommended_action", ""),
            "validation_status": "invalid",
            "raw_response": raw_response
        }

        insight = FollowupInsight(**data)
        if insight.root_cause in ALLOWED_CAUSES:
            result["validation_status"] = "valid"

    except (ValidationError, ValueError, TypeError, KeyError) as error:
        # Preserve the raw reply and any fields it contained.
        result = {
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": data.get("root_cause", ""),
            "confidence": data.get("confidence", ""),
            "summary": data.get("summary", ""),
            "recommended_action": data.get("recommended_action", ""),
            "validation_status": "invalid",
            "raw_response": raw_response
        }
        print(f"{row['note_id']} validation/JSON error: {error}")

    except Exception as error:
        # Keep the loop running if an API call fails.
        result = {
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": "",
            "confidence": "",
            "summary": "",
            "recommended_action": "",
            "validation_status": "invalid",
            "raw_response": str(error)
        }
        print(f"{row['note_id']} API error: {error}")

    results.append(result)

notes_analysed = pd.DataFrame(results, columns=[
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
])

notes_analysed.to_csv("notes_analysed.csv", index=False)

print("API calls made in this cell:", api_calls)
print("Rows saved:", len(notes_analysed))
print("Saved notes_analysed.csv")
print(notes_analysed[
    ["note_id", "root_cause", "confidence", "validation_status"]
].to_string(index=False))

API calls made in this cell: 9
Rows saved: 10
Saved notes_analysed.csv
note_id            root_cause  confidence validation_status
   N001 forgot_or_no_reminder        0.98             valid
   N002 distance_or_transport        0.95             valid
   N003           cost_or_fee        0.95             valid
   N004                 other        0.00     skipped_blank
   N005  illness_or_emergency        0.95             valid
   N006      work_or_schedule        0.95             valid
   N007 forgot_or_no_reminder        0.95             valid
   N008 distance_or_transport        0.95             valid
   N009                 other        0.90             valid
   N010      work_or_schedule        0.95             valid


In [7]:
from pydantic import ValidationError
import pandas as pd
import time
from google.genai import types

# (a) Test two deliberately invalid replies.
bad_reply_confidence = {
    "root_cause": "other",
    "confidence": 1.7,
    "summary": "The reason is unclear.",
    "recommended_action": "Ask the patient for more details."
}

bad_reply_missing_field = {
    "root_cause": "other",
    "confidence": 0.5,
    "summary": "The reason is unclear."
    # recommended_action is deliberately missing
}

for label, bad_reply in [
    ("Confidence above 1", bad_reply_confidence),
    ("Missing recommended_action", bad_reply_missing_field)
]:
    try:
        FollowupInsight(**bad_reply)
        print(label, "unexpectedly passed validation")
    except ValidationError as error:
        print(f"\n{label} validation error:")
        print(error)

# For the prompt comparison, keep system instructions and generation
# settings identical. Only the user prompt changes.
comparison_system = "Return JSON that follows the provided schema."

def generate_comparison_reply(prompt):
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=comparison_system,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )
    return response.text, json.loads(response.text)

comparison_results = []
api_calls_43 = 0

for note_id in ["N007", "N009"]:
    note_text = notes.loc[
        notes["note_id"] == note_id, "note_text"
    ].iloc[0]

    prompts = {
        "V1": f"Classify this note:\n\n{note_text}",
        "V2": build_craft_prompt(note_text)
    }

    for prompt_name, prompt in prompts.items():
        if api_calls_43 > 0:
            time.sleep(13)

        api_calls_43 += 1
        try:
            raw_text, data = generate_comparison_reply(prompt)
            cause = data.get("root_cause", "")
            comparison_results.append({
                "note_id": note_id,
                "prompt": prompt_name,
                "root_cause": cause,
                "allowed_value": cause in ALLOWED_CAUSES
            })
        except Exception as error:
            comparison_results.append({
                "note_id": note_id,
                "prompt": prompt_name,
                "root_cause": f"ERROR: {error}",
                "allowed_value": False
            })

comparison_table = pd.DataFrame(comparison_results)
print("\nPrompt comparison:")
print(comparison_table.to_string(index=False))
print("\nAPI calls in Part 4.3:", api_calls_43)


Confidence above 1 validation error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Missing recommended_action validation error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...The reason is unclear.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

Prompt comparison:
note_id prompt                 root_cause  allowed_value
   N007     V1   SMS notification failure          False
   N007     V2      forgot_or_no_reminder           True
   N009     V1 Patient phone switched off          False
   N009     V2                      other           True

API calls in Part 4.3: 4


In [8]:
import pandas as pd

clean_url = (
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/appointments_clean.csv"
)
clean_appointments = pd.read_csv(clean_url)
notes_analysed = pd.read_csv("notes_analysed.csv")

merged = notes_analysed.merge(
    clean_appointments[
        ["appointment_id", "reminder_sent", "status", "no_show"]
    ],
    on="appointment_id",
    how="left"
)

print("Merged rows:", len(merged))
print("\nNotes by root cause and reminder status:")
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"]))

print("\nReminder status counts:")
print(merged["reminder_sent"].value_counts(dropna=False))

Merged rows: 10

Notes by root cause and reminder status:
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1

Reminder status counts:
reminder_sent
No     5
Yes    5
Name: count, dtype: int64
